# Buổi 20 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `da_bien.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dầu và xăng: dừng hay không (mục 4.2)

Sửa `hang_dong_lien_ket` rồi chạy lại ô này.

In [ ]:
%matplotlib inline
import warnings

import da_bien as db
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
L = db.doc_dau_xang()
print(db.kiem_dung(L).round(3).to_string(index=False))
print(db.engle_granger(L))
hang = db.hang_dong_lien_ket(L)
ten, f = db.du_bao_xang(L, 4, hang)
print("hạng Johansen:", hang, "| mô hình dùng:", ten, "| giá xăng 4 tuần tới (USD/gallon):", np.exp(f).round(3))

## Bước 2 — Backtest VAR và VECM (mục 4.2)

In [ ]:
from tv.backtest import diebold_mariano

kq = db.backtest_dau_xang(L)
mae = kq.groupby("buoc_h")[["naive", "VAR", "VECM"]].apply(lambda d: d.sub(kq.loc[d.index, "y"], axis=0).abs().mean() * 100)
print(mae.round(2).to_string())
h4 = kq[kq["buoc_h"] == 4]
for m in ("VAR", "VECM"):
    dm = diebold_mariano((h4["y"] - h4[m]).to_numpy(), (h4["y"] - h4["naive"]).to_numpy(), h=4, ham_mat_mat="tuyet_doi")
    print(f"{m} so với naive, tầm 4 tuần: DM p = {dm.p_value:.2f}")

## Bước 3 — Kalman (mục 4.3)

In [ ]:
y = db.doc_nile()
ref = db.local_level_statsmodels(y)
s2_nhieu, s2_muc = ref.params
kq_k = db.kalman_local_level(y, s2_nhieu, s2_muc)
print("phương sai nhiễu, cú dời:", ref.params.round(0), "| lệch tối đa so statsmodels:", np.abs(kq_k["mức"] - ref.filtered_state[0]).max().round(3))
thieu = y.copy()
thieu[40:60] = np.nan                  # xoá số đo 1911–1930
kq_t = db.kalman_local_level(thieu, s2_nhieu, s2_muc)
print("độ lệch chuẩn của mức năm 1910 và 1930:", np.sqrt(kq_t["phương sai"][[39, 59]]).round(1))

## Bước 4 — Nowcast GDP (mục 4.5–4.6)

Lần đầu đọc tệp xlsx mất khoảng 15 giây. Sửa `nowcast` rồi chạy lại ô này.

In [ ]:
du_lieu = db.doc_tat_ca()
q = pd.Period("2008Q1")
for k in (1, 2, 3, 4):                 # ragged edge: mỗi chuỗi dừng ở đâu tại vintage giữa tháng thứ k
    v = db.vintage_cua(q, k)
    print(v, "| GDP tới", db.gdp_theo_vintage(du_lieu, v).index[-1],
          "| việc làm tới", db.chi_bao_theo_vintage(du_lieu, "e", v).index[-1])
bang = db.danh_gia_nowcast(du_lieu, pd.period_range("2005Q1", "2019Q4", freq="Q"))
print(db.rmse_theo_k(bang).round(2).to_string())

## Bước 5 — DFM (mục 4.4–4.5)

Khoảng 40 giây. Sau đó trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 6/6.

In [ ]:
bang_dfm = db.danh_gia_nowcast(du_lieu, pd.period_range("2005Q1", "2019Q4", freq="Q"), dfm=True)
print(db.rmse_theo_k(bang_dfm).round(2).to_string())